# Currency Converter Agent AI

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.messages import HumanMessage
from langchain.tools import tool
import requests
import os


c:\Users\shres\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
load_dotenv()
model=ChatOpenAI()



In [3]:
#Loading api from env
api_key=os.getenv("CURRENCY_API_KEY")

# Check if it loaded properly
if not api_key:
    print("❌ API key not found. Check your .env file.")
else:
    print("✅ API key loaded successfully!")
    print(f"Key starts with: {api_key[:5]}...")

✅ API key loaded successfully!
Key starts with: 717b2...


In [4]:
@tool 
def get_currency_factor(base_currency:str,target_currency:str,api_key:str =api_key)->float:
    """
    This function fetches the currency conversion factor between a given base factor and target currency
    """
    url=f"https://api.exchangeratesapi.io/v1/latest?access_key={api_key}&format=1"
    response=requests.get(url)
    return response.json()
    

In [5]:
get_currency_factor.invoke({'base_currency':'USD','target_currency':'NPR'})

{'success': True,
 'timestamp': 1773651968,
 'base': 'EUR',
 'date': '2026-03-16',
 'rates': {'AED': 4.201877,
  'AFN': 72.080993,
  'ALL': 95.933352,
  'AMD': 432.101693,
  'ANG': 2.048115,
  'AOA': 1049.180577,
  'ARS': 1599.843022,
  'AUD': 1.630533,
  'AWG': 2.059461,
  'AZN': 1.946563,
  'BAM': 1.954442,
  'BBD': 2.308296,
  'BDT': 140.632298,
  'BGN': 1.955698,
  'BHD': 0.432023,
  'BIF': 3402.336281,
  'BMD': 1.144145,
  'BND': 1.466581,
  'BOB': 7.919498,
  'BRL': 6.142489,
  'BSD': 1.146104,
  'BTC': 1.5655978e-05,
  'BTN': 105.788967,
  'BWP': 15.617218,
  'BYN': 3.391144,
  'BYR': 22425.244418,
  'BZD': 2.304899,
  'CAD': 1.567462,
  'CDF': 2582.335529,
  'CHF': 0.903566,
  'CLF': 0.026678,
  'CLP': 1053.402843,
  'CNY': 7.890712,
  'CNH': 7.89685,
  'COP': 4227.639114,
  'CRC': 539.227737,
  'CUC': 1.144145,
  'CUP': 30.319846,
  'CVE': 110.18893,
  'CZK': 24.448433,
  'DJF': 204.088213,
  'DKK': 7.471748,
  'DOP': 70.411391,
  'DZD': 151.571795,
  'EGP': 59.964521,
  'ERN'

In [47]:
#Creating conversion function
@tool
def conversion_rate(base_currency:str,target_currency:str):
    """
    This tool only fetches exchange rates.
    Use the currency_convert tool afterwards to calculate conversion.
    
    """
    
    api_key=os.getenv("CURRENCY_API_KEY")
    if not api_key:
        return {
            "success": False,
            "error": "API key not configured"
        }
    
    # Make API call
    url = f"https://api.exchangeratesapi.io/v1/latest?access_key={api_key}&format=1"
    
    try:
        response=requests.get(url)
        data = response.json()
        if data.get('success'):
            rates=data.get('rates',{})
            
            #Convert currency string into upper case
            base_currency=base_currency.upper()
            target_currency=target_currency.upper()
            
            #Check wheather the currency rate for base and target currency exist  on the list
            if base_currency not in rates:
                return{
                    "success": False,
                        "error": f"Currency '{base_currency}' not found",
                        "available_currencies": list(rates.keys())[:10] 
                }
                
            if target_currency not in rates:
                return{
                    "success": False,
                        "error": f"Currency '{target_currency}' not found",
                        "available_currencies": list(rates.keys())[:10] 
                }
            # Calculate conversion
            base_currency_rate = rates[base_currency]
            target_currency_rate = rates[target_currency]
                
            return base_currency_rate ,target_currency_rate
                
        else:
            return {
                    "success": False,
                    "error": data.get('error', {}).get('message', 'Unknown error')
                }
                                                                                       
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }
@tool
def currency_convert(base_currency_rate:int,target_currency_rate:int)->float:
    """    Convert currency using provided exchange rates.

    """
    amount=1.0
    #Conversion rate calculation
    base_rate=base_currency_rate
    target_rate=target_currency_rate
    
    # conversion rate formula
    conversion_rate=target_rate/base_rate
    converted_amount=amount*conversion_rate
        
    return converted_amount
    

In [48]:
#tool_binding
llm_with_tools=model.bind_tools([conversion_rate,currency_convert])

In [52]:
messages=[HumanMessage("what is the conversion factor for the USD and NPR, and based on that can you convert 10 USD into NPR  ")]

In [53]:
AI_message=llm_with_tools.invoke(messages)
AI_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 117, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DJzE4EiEsl2YU4ZUnN5QIN8kBYTPt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf622-1608-7280-86bd-ef7d46de2172-0', tool_calls=[{'name': 'conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_QxG8kgVmHr9UhBQrsoxWze4s', 'type': 'tool_call'}, {'name': 'currency_convert', 'args': {'base_currency_rate': 1, 'target_currency_rate': 115.2}, 'id': 'call_Ng1xhRhB2EHfJxGGZzGqNJ04', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

In [54]:
AI_message.tool_calls

[{'name': 'conversion_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_QxG8kgVmHr9UhBQrsoxWze4s',
  'type': 'tool_call'},
 {'name': 'currency_convert',
  'args': {'base_currency_rate': 1, 'target_currency_rate': 115.2},
  'id': 'call_Ng1xhRhB2EHfJxGGZzGqNJ04',
  'type': 'tool_call'}]